<a href="https://colab.research.google.com/github/rudraroy1555/resume-screener-ranking/blob/main/tf_idf_resume_screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 1. The Target: What are you hiring for?
job_description = """
Looking for a software engineer with strong Python skills.
Must have experience with data analysis, pandas, and machine learning.
"""

# 2. The Candidates: Your copy-pasted raw text
resumes_data = [
    {
        "candidate_name": "Alice",
        "raw_text": "I am a software engineer. I code in Python every day. I use pandas for data analysis and build machine learning models."
    },
    {
        "candidate_name": "Bob",
        "raw_text": "Frontend web developer. I write HTML, CSS, and JavaScript. I build user interfaces in React."
    },
    {
        "candidate_name": "Charlie",
        "raw_text": "Data analyst with heavy python experience. I clean data using pandas and know basic machine learning."
    }
]

# 3. Convert to a DataFrame so we can manipulate it easily
df = pd.DataFrame(resumes_data)

print("--- Job Description ---")
print(job_description.strip())
print("\n--- Candidate DataFrame ---")
print(df)

--- Job Description ---
Looking for a software engineer with strong Python skills. 
Must have experience with data analysis, pandas, and machine learning.

--- Candidate DataFrame ---
  candidate_name                                           raw_text
0          Alice  I am a software engineer. I code in Python eve...
1            Bob  Frontend web developer. I write HTML, CSS, and...
2        Charlie  Data analyst with heavy python experience. I c...


In [ ]:
# Download all required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
#nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
def clean_and_lemmatize(text):
    """Strips punctuation, lowercases, tokenizes with punkt, removes stopwords, and lemmatizes."""
    # 1. Lowercase
    text = text.lower()

    # 2. Regex to remove punctuation (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # 3. Tokenize using punkt instead of .split()
    words = word_tokenize(text)

    # 4. Setup NLP tools
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    # 5. Filter and lemmatize
    clean_words = []
    for word in words:
        if word not in stop_words:
            clean_words.append(lemmatizer.lemmatize(word))

    return ' '.join(clean_words)

# Apply to JD and DataFrame
clean_jd = clean_and_lemmatize(job_description)
df['clean_text'] = df['raw_text'].apply(clean_and_lemmatize)

print("--- Cleaned Text ---")
print(df[['candidate_name', 'clean_text']])

--- Cleaned Text ---
  candidate_name                                         clean_text
0          Alice  software engineer code python every day use pa...
1            Bob  frontend web developer write html cs javascrip...
2        Charlie  data analyst heavy python experience clean dat...


In [ ]:
# 1. Initialize the vectorizer
vectorizer = TfidfVectorizer()

# 2. Combine all text
all_text = [clean_jd] + df['clean_text'].tolist()

# 3. Fit and transform
tfidf_matrix = vectorizer.fit_transform(all_text)

# 4. Create DataFrame to view the matrix
feature_names = vectorizer.get_feature_names_out()
dense_matrix = tfidf_matrix.toarray()

matrix_df = pd.DataFrame(dense_matrix, columns=feature_names)
matrix_df.index = ['Job Description', 'Alice', 'Bob', 'Charlie']

print("-----------------TF-IDF Matrix----------------")
# .T swaps rows and columns so you can read the words down the left side
print(matrix_df.round(3).T)

-----------------TF-IDF Matrix----------------
            Job Description  Alice    Bob  Charlie
analysis              0.270  0.255  0.000    0.000
analyst               0.000  0.000  0.000    0.318
basic                 0.000  0.000  0.000    0.318
build                 0.000  0.255  0.242    0.000
clean                 0.000  0.000  0.000    0.318
code                  0.000  0.324  0.000    0.000
cs                    0.000  0.000  0.307    0.000
data                  0.219  0.207  0.000    0.406
day                   0.000  0.324  0.000    0.000
developer             0.000  0.000  0.307    0.000
engineer              0.270  0.255  0.000    0.000
every                 0.000  0.324  0.000    0.000
experience            0.270  0.000  0.000    0.251
frontend              0.000  0.000  0.307    0.000
heavy                 0.000  0.000  0.000    0.318
html                  0.000  0.000  0.307    0.000
interface             0.000  0.000  0.307    0.000
javascript            0.000  0.000 

In [ ]:
# 1. Isolate the vectors
# The Job Description is the first row (index 0)
vector = tfidf_matrix[0]

# The candidates are everything from row 1 to the end
candidate_vectors = tfidf_matrix[1:]

# 2. Calculate the similarity scores
# This compares the JD vector against EVERY candidate vector at once
scores = cosine_similarity(vector, candidate_vectors)

# .flatten() converts a 2D array [[0.8, 0.1, 0.6]] into a 1D list [0.8, 0.1, 0.6]
flat_scores = scores.flatten()

# 3. Add the scores back to our main DataFrame
df['similarity_score'] = flat_scores

# 4. Sort the DataFrame from highest score to lowest
df_ranked = df.sort_values(by='similarity_score', ascending=False)

print("--- Final Candidate Ranking ---")
# Print just the relevant columns
print(df_ranked[['candidate_name', 'similarity_score', 'raw_text']])

--- Final Candidate Ranking ---
  candidate_name  similarity_score  \
0          Alice          0.433077   
2        Charlie          0.334099   
1            Bob          0.000000   

                                            raw_text  
0  I am a software engineer. I code in Python eve...  
2  Data analyst with heavy python experience. I c...  
1  Frontend web developer. I write HTML, CSS, and...  


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

exp_df = df_ranked[['candidate_name', 'similarity_score', 'raw_text']]
path = "/content/drive/MyDrive/candidate_ranking_results.csv"

# 4. Save the cleanly formatted dataframe to the path
exp_df.to_csv(path, index=False)

print("Data successfully cleaned and exported to Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data successfully cleaned and exported to Google Drive.


In [ ]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def test_pipeline(job_desc, resume_text):
    """Runs a single resume against a job description and returns the similarity score."""

    def clean_text(text):
        text = text.lower()
        text = re.sub(r'[^a-z\s]', '', text)
        words = word_tokenize(text)
        stop_words = set(stopwords.words('english'))
        lemmatizer = WordNetLemmatizer()
        clean_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
        return ' '.join(clean_words)

    # 1. Clean both inputs
    clean_jd = clean_text(job_desc)
    clean_resume = clean_text(resume_text)

    # 2. Vectorize
    vectorizer = TfidfVectorizer()
    # We fit on both so the vocabulary includes words from both texts
    tfidf_matrix = vectorizer.fit_transform([clean_jd, clean_resume])

    # 3. Score
    jd_vector = tfidf_matrix[0]
    resume_vector = tfidf_matrix[1]

    # Calculate cosine similarity between the two vectors
    score = cosine_similarity(jd_vector, resume_vector).flatten()[0]

    print("\n--- Pipeline Test Results ---")
    print(f"Similarity Score: {score:.4f} (0.0 to 1.0)")
    print("-" * 27)
    return score

# ==========================================
# TEST YOUR INPUTS HERE
# ==========================================

my_job_description = """
Looking for a senior backend engineer. Must know Node.js, Express, and MongoDB.
Experience with AWS and Docker is required.
"""

my_test_resume = """
I am a backend developer. I have 5 years of experience building APIs with Node.js and Express.
I deploy my applications using Docker containers on AWS. I also manage MongoDB databases."""

# Run the test
test_pipeline(my_job_description, my_test_resume)


--- Pipeline Test Results ---
Similarity Score: 0.3008 (0.0 to 1.0)
---------------------------


np.float64(0.30078887295984286)